In [1]:
# Menyiapkan Dataset

In [2]:
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20])
}

df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/mahasiswa/tugas4/
print("Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, sum as spark_sum, count, avg

spark = SparkSession.builder \
    .appName("TugasMandiri4") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

26/09/16 10:19:08 WARN Utils: Your hostname, hans-Nitro-AN515-58 resolves to a loopback address: 127.0.1.1; using 192.168.43.220 instead (on interface wlp0s20f3)
26/09/16 10:19:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/16 10:19:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
# A.Membaca dan Eksplorasi Awal (Code Cell)

In [5]:
path_hdfs = "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv"
df = spark.read.csv(path_hdfs, header=True, inferSchema=True)

print("--- Schema Data ---")
df.printSchema()

print("--- Jumlah Baris ---")
print("Total baris:", df.count())

print("--- 10 Baris Pertama ---")
df.show(10)

--- Schema Data ---
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

--- Jumlah Baris ---
Total baris: 1000
--- 10 Baris Pertama ---
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-

In [6]:
# B. Menangani Data Kosong

In [7]:
# Menghitung jumlah data null pada kolom rating
null_count = df.filter(col("rating").isNull()).count()
print("Jumlah baris dengan rating kosong (null):", null_count)

# Mengisi data rating yang kosong dengan nilai 0
df_clean = df.na.fill({"rating": 0})

Jumlah baris dengan rating kosong (null): 204


Penanganan dilakukan menggunakan fill(0) agar baris transaksi yang tidak memiliki rating tetap tersimpan. Menghapus baris dengan drop() akan menghilangkan data kuantitatif penting lainnya seperti unit_terjual dan harga_satuan, yang berisiko mengacaukan perhitungan total pendapatan bisnis.

In [8]:
# C. Transformasi Data 

In [9]:
df_transformed = df_clean.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan")) \
                         .withColumn("tier_transaksi", when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil"))

df_transformed.show(10)

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         Kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         Besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         Kecil|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|         21

In [10]:
# D. Analisis dengan GroupBy

In [11]:
# 1. Kategori dengan total_pendapatan tertinggi
print("--- 1. Kategori Pendapatan Tertinggi ---")
df_transformed.groupBy("kategori") \
    .agg(spark_sum("total_pendapatan").alias("total_pendapatan")) \
    .orderBy(col("total_pendapatan").desc()) \
    .show(1)

# 2. Kota dengan jumlah transaksi tier "Besar" terbanyak
print("--- 2. Kota Transaksi Tier 'Besar' Terbanyak ---")
df_transformed.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(1)

# 3. Rata-rata rating untuk masing-masing metode_pembayaran
print("--- 3. Rata-rata Rating per Metode Pembayaran ---")
df_transformed.groupBy("metode_pembayaran") \
    .agg(avg("rating").alias("rata_rata_rating")) \
    .orderBy(col("rata_rata_rating").desc()) \
    .show()

--- 1. Kategori Pendapatan Tertinggi ---
+------------+----------------+
|    kategori|total_pendapatan|
+------------+----------------+
|Rumah Tangga|       138665000|
+------------+----------------+
only showing top 1 row

--- 2. Kota Transaksi Tier 'Besar' Terbanyak ---
+----+-----+
|kota|count|
+----+-----+
|Solo|   92|
+----+-----+
only showing top 1 row

--- 3. Rata-rata Rating per Metode Pembayaran ---
+-----------------+------------------+
|metode_pembayaran|  rata_rata_rating|
+-----------------+------------------+
|              COD|3.3745019920318726|
|    Transfer Bank|3.3399209486166006|
|         E-Wallet|             3.292|
|     Kartu Kredit|3.1910569105691056|
+-----------------+------------------+



In [12]:
# E. Menyimpan Hasil ke HDFS 

In [13]:
output_hdfs = "hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_transaksi_september"

# Menyimpan DataFrame ke HDFS
df_transformed.write.csv(output_hdfs, header=True, mode="overwrite")

# Verifikasi hasil simpan dari HDFS
df_verifikasi = spark.read.csv(output_hdfs, header=True, inferSchema=True)
print("Jumlah baris hasil terverifikasi:", df_verifikasi.count())
df_verifikasi.show(5)

Jumlah baris hasil terverifikasi: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         Kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         Besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         Kecil|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      3500

Spark bekerja secara komputasi terdistribusi dengan membagi data ke dalam beberapa partisi memori. Saat proses penulisan (write) berlangsung, tiap worker thread / partisi mengeksekusi penulisan file secara paralel ke HDFS. Hal ini menyebabkan luaran tersimpan sebagai direktori yang berisi beberapa berkas partisi (part-00000...), bukan satu file CSV tunggal, guna memaksimalkan kecepatan proses eksekusi data berskala besar.